In [3]:
import sqlite3
import pandas as pd


In [ ]:
# create or connect to a database file called "potter_airlines.db"
# if "potter_airlines.db"does not exist, SQLite create a new database file with this name
conn = sqlite3.connect("potter_airlines.db")

# create a cursor
# cursor is what we use to send SQL commends to SQLite

cursor = conn.cursor()

# tell SQLite to excute the SQL code inside.
# create the table
cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    flight_id TEXT PRIMARY KEY,
    origin TEXT NOT NULL,
    destination TEXT NOT NULL,
    departure_date TEXT NOT NULL,
    economy_base_fare REAL NOT NULL,
    economy_seats_remaining INTEGER NOT NULL,
    economy_capacity INTEGER NOT NULL,
    business_base_fare REAL NOT NULL,
    business_seats_remaining INTEGER NOT NULL,
    business_capacity INTEGER NOT NULL
)
""")

# create unique index prevent duplicate flight records

cursor.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS unique_flight
ON flights (flight_id, departure_date)
""")

# confirm and save the changes
conn.commit()

# close the connection to SQLite
conn.close()

In [ ]:
df = pd.read_csv("flights.csv")

# Standardize the date format to prevent duplicate records.
df["departure_date"] = pd.to_datetime(
    df["departure_date"]
).dt.strftime("%Y-%m-%d")

In [ ]:
# insert data information into the database

conn = sqlite3.connect("potter_airlines.db")
cursor = conn.cursor()

# iterate through the rows of the DataFrame
for _, row in df.iterrows():
    cursor.execute("""
        INSERT OR IGNORE INTO flights (
            flight_id,
            origin,
            destination,
            departure_date,
            economy_base_fare,
            economy_seats_remaining,
            economy_capacity,
            business_base_fare,
            business_seats_remaining,
            business_capacity
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["flight_id"],
        row["origin"],
        row["destination"],
        row["departure_date"],
        row["economy_base_fare"],
        row["economy_seats_remaining"],
        row["economy_capacity"],
        row["business_base_fare"],
        row["business_seats_remaining"],
        row["business_capacity"]
    ))

conn.commit()
conn.close()



In [ ]:
# DELETE duplicated records in the flights table
conn = sqlite3.connect("potter_airlines.db")
cursor = conn.cursor()

cursor.execute("""
DELETE FROM flights
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM flights
    GROUP BY flight_id, departure_date
)
""")

conn.commit()
conn.close()

In [ ]:
# the demo search function with SQLite

def search_flights(origin, destination, cabin_class):

    conn = sqlite3.connect("potter_airlines.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT flight_id, departure_date, origin, destination, 
        economy_base_fare, economy_seats_remaining, economy_capacity,
        business_base_fare, business_seats_remaining, business_capacity
        FROM flights
        WHERE origin = ?
        AND destination = ?
    """, (
        origin,
        destination
    ))

    results = cursor.fetchall()

    conn.close()

    output = []

    for flight in results:

        if cabin_class.lower() == "economy":
            base_fare = flight[4]
            seats_remaining = flight[5]
            capacity = flight[6]

        elif cabin_class.lower() == "business":
            base_fare = flight[7]
            seats_remaining = flight[8]
            capacity = flight[9]

        else:
            return "Invalid cabin class."

        seat_factor = 1 + (seats_remaining / capacity)
        max_price = base_fare * 2
        price = base_fare * seat_factor
        min_price = base_fare * 0.8
        final_price = max(min_price, min(price, max_price))

        output.append((
            flight[0],
            flight[1],
            flight[2],
            flight[3],
            cabin_class,
            f"${final_price:.2f}"
        ))


    return output

In [ ]:
# searching function testing

search_flights(
        "Toronto",
        "London",
        "Business"
    )


[('PA1002', '2026-10-22', 'Toronto', 'London', 'Business', '$1097.23'),
 ('PA1002', '2026-12-11', 'Toronto', 'London', 'Business', '$685.77')]

In [ ]:
# the demo booking functin with SQLite

def book_flight(
    flight_id,
    departure_date,
    origin,
    destination,
    cabin_class,
    tickets
):

    conn = sqlite3.connect("potter_airlines.db")
    cursor = conn.cursor()

    if cabin_class.lower() == "economy":
        seat_column = "economy_seats_remaining"

    elif cabin_class.lower() == "business":
        seat_column = "business_seats_remaining"

    else:
        conn.close()
        return "Invalid cabin class."

    cursor.execute(
        f"""
        SELECT {seat_column}
        FROM flights
        WHERE flight_id = ?
        AND departure_date = ?
        AND origin = ?
        AND destination = ?
        """,
        (
            flight_id,
            departure_date,
            origin,
            destination
        )
    )

    result = cursor.fetchone()

    if result is None:
        conn.close()
        return "Flight information does not match."

    seats_remaining = result[0]

    if tickets <= 0:
        conn.close()
        return "Invalid number of tickets."

    if tickets > seats_remaining:
        conn.close()
        return "Not enough seats available."

    cursor.execute(
        f"""
        UPDATE flights
        SET {seat_column} = {seat_column} - ?
        WHERE flight_id = ?
        AND departure_date = ?
        AND origin = ?
        AND destination = ?
        """,
        (
            tickets,
            flight_id,
            departure_date,
            origin,
            destination
        )
    )

    conn.commit()
    conn.close()

    return "Booking successful."

In [ ]:
# booking function testing

book_flight(
        "PA1002",
        "2026-10-22",
        "Toronto",
        "London",
        "Business",
        2
    )


'Booking successful.'

In [ ]:
# checking remaining seats -- testing code -- whether the booking function is working properly
conn = sqlite3.connect("potter_airlines.db")
cursor = conn.cursor()

cursor.execute("""
    SELECT flight_id,
           departure_date,
           economy_seats_remaining,
           business_seats_remaining
    FROM flights
    WHERE flight_id = ?
    AND departure_date = ?
""", (
    "PA1002",
    "2026-10-22"
))

result = cursor.fetchone()

print(result)

conn.close()

In [ ]:
# booking function testing -- no enough seats
book_flight(
        "PA1002",
        "2026-10-22",
        "Toronto",
        "London",
        "Business",
        20
    )


'Not enough seats available.'